# nb_coord_silver_arms_ingest
## Coordinator: ARMS Dataverse -> Silver Cleansed (Direct)

Metadata-driven ingestion from Synapse Link for Dataverse for ARMS.
Reads entity list and Dataverse container path from the metadata control table,
applies snake_case conversion, optional JSON validation, and writes to Silver cleansed.

- **ARMS** (step=`silver_arms_direct_ingest`, binding_id=12)



## Run Common Shared Notebooks

In [14]:
%run "/05 - shared-libraries/nb_shared_common"

StatementMeta(DataWHv35, 7628, 38, Finished, Available, Finished, False)

In [15]:
%run "/05 - shared-libraries/nb_shared_metadata"

StatementMeta(DataWHv35, 7628, 50, Finished, Available, Finished, False)

In [16]:
%run "/05 - shared-libraries/nb_shared_logging"

StatementMeta(DataWHv35, 7628, 53, Finished, Available, Finished, False)

In [17]:
# Pipeline parameters
storage_account_name = ""
batch_run_id = ""
run_date = ""
pipeline_id = ""
parent_pipeline_id = ""
top_level_pipeline_id = ""
pipeline_name=""
retry_count = 0

StatementMeta(DataWHv35, 7628, 54, Finished, Available, Finished, False)

In [18]:
# This cell is generated from runtime parameters. Learn more: https://go.microsoft.com/fwlink/?linkid=2161015
batch_run_id = "run_20261102T0007"
run_date = "20261102T0007"
pipeline_id = "89460948-7c79-459c-a20b-9d360029e56a"
parent_pipeline_id = "dcca7fd5-f8a0-41a7-969f-d507236a7ce8"
top_level_pipeline_id = "34f7c918-7ce7-42a9-a4ee-b46ce96dd74f"
storage_account_name = "deintdlsaueaasqebla"
pipeline_name = "pl_coordinator_silver_arms_ingest"
retry_count = 2
 


StatementMeta(DataWHv35, 7628, 55, Finished, Available, Finished, False)

## Import Libraries and setup data lake paths

In [19]:
#import libraries
from mdw_shared import audit, metadata, transforms
from mdw_shared.audit import *
from mdw_shared.metadata import *
from mdw_shared.transforms import *
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, trim, lower, regexp_replace, max as spark_max, to_timestamp
from mdw_shared.transforms import drop_and_register_table
from delta.tables import DeltaTable

# Get spark session and delta lake paths
spark = get_spark_session()

data_lake_paths = get_datalake_paths(storage_account_name)

BRONZE_BASE_PATH = data_lake_paths["bronze"]
SILVER_BASE_PATH = data_lake_paths["silver"]
GOLD_BASE_PATH = data_lake_paths["gold"]
METADATA_BASE_PATH = data_lake_paths["metadata"]

METADATA_CONTROL_PATH = f"{METADATA_BASE_PATH}/control"
METADATA_AUDIT_PATH = f"{METADATA_BASE_PATH}/audit"
METADATA_CONFIG_PATH = f"{METADATA_BASE_PATH}/config/silver/cleanse"

STEP_NAME_TO_RUN = STEP_SILVER_ARMS_DIRECT_INGEST
PIPELINE_NAME = pipeline_name
NOTEBOOK_NAME = "nb_coord_silver_arms_ingest"

QUARANTINE_BASE_PATH = f"{SILVER_BASE_PATH}/quarantine"

StatementMeta(DataWHv35, 7628, 56, Finished, Available, Finished, False)

## Get ingestion config from metadata and ingest each entity

In [20]:
# variables declarations
total_quarantined = 0
# store exception as it will be used at the end of the notebook
captured_exception = None

try:

    # Log Batch Start
    log_batch_event(METADATA_AUDIT_PATH,
                    batch_run_id=batch_run_id,
                    run_date=run_date,
                    retry_count=retry_count,
                    event_type=EVENT_TYPE_STARTED,
                    status=EVENT_STATUS_RUNNING,
                    pipeline_name=PIPELINE_NAME,
                    pipeline_id=pipeline_id,
                    parent_pipeline_id=parent_pipeline_id,
                    top_level_pipeline_id=top_level_pipeline_id,
                    phase=EVENT_PHASE_CLEANSE_ARMS
    )

    
    # =============================================================================
    # Get the ingestion mapping for ARMS to get the watermark values
    # =============================================================================
    ingestion_configs = get_silver_ingestion_config(METADATA_CONTROL_PATH, ARMS_SOURCE_SYSTEM)
    if ingestion_configs is None:
        raise Exception(
                f"Ingest configs for ARMS watermarking not found for Metadata Control Path: "
                f"{METADATA_CONTROL_PATH}, Step Name to run {STEP_NAME_TO_RUN}")

    print(f"Found {len(ingestion_configs)} tables to ingest from {ARMS_SOURCE_SYSTEM}")

    # =============================================================================
    # Get all ARMS cleanse entity mappings from metadata based on the STEP_NAME_TO_RUN
    # =============================================================================
    transform_configs = get_transform_config(METADATA_CONTROL_PATH, STEP_NAME_TO_RUN)
    if not transform_configs:
        raise Exception(
                f"Transform configs not found for Metadata Control Path: "
                f"{METADATA_CONTROL_PATH}, Step Name to run {STEP_NAME_TO_RUN}"
                    )

    # =============================================================================
    # For each ARMS entity configured in the metadata, read bronze -> cleanse -> write silver cleansed
    # =============================================================================
    for cfg in transform_configs:

        entity_name = cfg.target_entity_name.lower()
        source_entity = cfg.source_entity_name
        target_path = cfg.target_path.lower()
        
        _activityName=entity_name
        ingest_cfg = next(
            (
                icfg 
                for icfg in ingestion_configs 
                if icfg.source_object_name == source_entity
                and icfg.target_name == entity_name
            ),
            None
        )

        # we need to get ingestion watermark value per data to only bring across the data from ARMS since the last time we ran
        ingestion_mapping_id = ingest_cfg.ingestion_mapping_id if ingest_cfg else None
        ingestion_mode = (ingest_cfg.ingestion_mode if ingest_cfg else "full").lower()
        watermark_column = ingest_cfg.watermark_column if ingest_cfg else None
        last_success_watermark_value = ingest_cfg.last_success_watermark_value if ingest_cfg else None

        # Source path is now resolved from the metadata control table
        source_path = cfg.source_path.lower() # datavarse path is in lower case 
        _activity_name=f"arms_ingest_{entity_name}"

        # Log activity start.  Returns the activity id and act_start to use later when logging activity end
        act_id, act_start = log_activity_started(METADATA_AUDIT_PATH,
                                                batch_run_id=batch_run_id,
                                                activity_name=_activityName,
                                                notebook_name=NOTEBOOK_NAME,
                                                source_entity_name=source_entity,
                                                target_entity_name=entity_name,
                                                source_layer=EVENT_PHASE_BRONZE_ARMS,
                                                target_layer=EVENT_PHASE_CLEANSE_ARMS,
                                                retry_count=retry_count,
                                                pipeline_id=pipeline_id
        )

        try:
            #Read from Dataverse Delta table (path from metadata)
            df = spark.read.format("delta").load(source_path)
           
            if ingestion_mode == "incremental":
                if watermark_column not in df.columns:
                    print(
                        f"{entity_name}: Watermark column '{watermark_column}' not found in source '{source_entity}'. "
                        f"Processing full entity snapshot."
                    )
                elif not last_success_watermark_value:
                    print(
                        f"{entity_name}: No previous successful watermark value found. "
                        f"This is likely the first incremental run. Processing all available records."
                    )
                else:
                    wm_type = str(df.schema[watermark_column].dataType)
                    wm_value = str(last_success_watermark_value)

                    incremental_filter = col(watermark_column) > lit(wm_value)
                    if "TimestampType" in wm_type or "DateType" in wm_type:
                        incremental_filter = col(watermark_column) > to_timestamp(lit(wm_value))
                    df = df.filter(incremental_filter)

            rows_read = df.count()

            # Preserve pre-transform dataset for watermark max calculation
            watermark_df = df

            # if ingestion mode is 'incremental' and no rows to process, log activity succeeded and move onto next entity
            if ingestion_mode == "incremental" and rows_read == 0:
                if ingestion_mapping_id is not None and watermark_column:
                    update_watermark(METADATA_CONTROL_PATH, ingestion_mapping_id, None)

                log_activity_completed(METADATA_AUDIT_PATH,
                                activity_run_id=act_id,
                                batch_run_id=batch_run_id,
                                activity_name=_activityName,
                                status=EVENT_STATUS_SUCCEEDED,
                                notebook_name=NOTEBOOK_NAME,
                                source_entity_name=source_entity,
                                target_entity_name=entity_name,
                                source_layer=EVENT_PHASE_BRONZE_ARMS,
                                target_layer=EVENT_PHASE_CLEANSE_ARMS,
                                retry_count=retry_count,
                                pipeline_id=pipeline_id,
                                started_at=act_start,
                                delta_version=None
                )

                print(f"  ~ {entity_name}: no incremental changes")
                continue

            #  *** We have rows to process ***
            rules = load_validation_rules(f"{ARMS_SOURCE_SYSTEM}/{entity_name}", metadata_config_path=METADATA_CONFIG_PATH)        

            # Generic error message either json file is not exist or malformed
            if not rules:
                raise Exception(
                    f"Either no configuration found OR configuration malformed for entity '{entity_name}' "
                    f" in metadata configuration path='{METADATA_CONFIG_PATH}'"
                )

            entity_quarantined = 0
            rules_applied = 0
            if rules and "column_mappings" in rules:            
                df = apply_column_mappings(df, rules["column_mappings"])
                        
            if rules and "validations" in rules:
                rules_applied = len(rules["validations"])            
                df, invalid_df = apply_validations(df, rules["validations"])
                
                # Write invalid rows to quarantine for investigation
                q_count = write_quarantine(invalid_df, QUARANTINE_BASE_PATH, entity_name, batch_run_id)
                if q_count > 0:
                    entity_quarantined = q_count
                    total_quarantined += q_count
                    print(f"  ! {entity_name}: {q_count} invalid rows quarantined")

            # Standardise text fields (trim, normalise whitespace)
            string_cols = [f.name for f in df.schema.fields if str(f.dataType) == 'StringType']
            for c in string_cols:
                df = df.withColumn(c, trim(regexp_replace(col(c), r'\s+', ' ')))

            # Deduplicate if primary key defined
            if rules and "primary_key" in rules:
                primary_key_col_name_arr = rules["primary_key"]
                order_by_col = try_get_col_name(df, ["modifiedon", "modified_on", "SinkModifiedOn", "sink_modified_on"])
                if order_by_col is not None:
                    df = deduplicate(df, primary_key_col_name_arr, order_by_col)
                
                assert_no_duplicates(df, primary_key_col_name_arr, source_entity)   
            
            # 2c. Source columns may differ with Target Columns in Cleanse or Source Columns may be NULL . 
            # Following code does the type casting to all data types to target schema
            target_schema = {f.name.lower(): f.dataType for f in spark.read.format("delta").load(target_path).schema.fields}
            for field in df.schema.fields:
                existing_type = target_schema.get(field.name.lower())
                if existing_type and existing_type != field.dataType:
                    df = df.withColumn(field.name, col(field.name).cast(existing_type))

            # Add silver metadata columns
            df = derive_change_type(df)
            df = add_silver_transaction_metadata(df, batch_run_id, act_id, source_system=ARMS_SOURCE_SYSTEM)
            df = df.withColumn("mdw_source_entity", lit(source_entity))

            rows_written = df.count()
            
            # Idempotent write: delete any existing rows from a prior run of this
            # batch before appending, preventing duplicates on re-runs.
            delete_batch_data(spark, target_path, batch_run_id)

            # Write as Delta (append with schema evolution for new source columns)
            df.write.format("delta").mode("append").option("mergeSchema", "true").save(target_path)

            # Capture Delta version after write for time travel traceability
            target_version = capture_delta_version(spark, target_path)

            if ingestion_mapping_id is not None and watermark_column:
                max_wm = None
                if watermark_column in watermark_df.columns:
                    max_wm = watermark_df.agg(spark_max(col(watermark_column))).collect()[0][0]
                update_watermark(METADATA_CONTROL_PATH, ingestion_mapping_id, max_wm)

            # Configure Delta retention on first successful write
            ensure_delta_retention(spark, target_path)

            # Log data quality metrics
            rows_valid = rows_written  # rows that passed validation
            log_data_quality(METADATA_AUDIT_PATH,
                            batch_run_id=batch_run_id,
                            entity_name=entity_name,
                            layer=EVENT_PHASE_CLEANSE_ARMS,
                            rows_read=rows_read,
                            rows_valid=rows_valid,
                            rows_quarantined=entity_quarantined,
                            rows_written=rows_written,
                            validation_rules_applied=rules_applied,
                            notebook_name=NOTEBOOK_NAME,
                            activity_run_id=act_id,
                            pipeline_name=PIPELINE_NAME
            )

            # Log lineage event (source -> target)
            source_version = capture_delta_version(spark, source_path)
            log_lineage_event(METADATA_AUDIT_PATH,
                batch_run_id=batch_run_id,
                source_entity=source_entity,
                source_layer=EVENT_PHASE_BRONZE_ARMS,
                target_layer=EVENT_PHASE_CLEANSE_ARMS,
                target_entity=entity_name,
                activity_run_id=act_id,
                source_path=source_path, target_path=target_path,
                transform_type=EVENT_PHASE_CLEANSE_ARMS,
                rows_in=rows_read, rows_out=rows_written,
                delta_version_source=source_version,
                delta_version_target=target_version,
                notebook_name=NOTEBOOK_NAME,
                pipeline_name=PIPELINE_NAME
            )

            log_activity_completed(METADATA_AUDIT_PATH,
                                activity_run_id=act_id,
                                batch_run_id=batch_run_id,
                                activity_name=_activityName,
                                status=EVENT_STATUS_SUCCEEDED,
                                notebook_name=NOTEBOOK_NAME,
                                source_entity_name=source_entity,
                                target_entity_name=entity_name,
                                source_layer=EVENT_PHASE_BRONZE_ARMS,
                                target_layer=EVENT_PHASE_CLEANSE_ARMS,
                                rows_read=rows_read,
                                rows_written=rows_written,
                                retry_count=retry_count,
                                pipeline_id=pipeline_id,
                                started_at=act_start,
                                delta_version=target_version,
                                rows_quarantined=entity_quarantined
            )

        except Exception as e:
            message = f"ARMS cleanse has failed for entity: {entity_name}"

            log_activity_completed(METADATA_AUDIT_PATH,
                                activity_run_id=act_id,
                                batch_run_id=batch_run_id,
                                activity_name=_activityName,
                                status=EVENT_STATUS_FAILED,
                                notebook_name=NOTEBOOK_NAME,
                                source_entity_name=source_entity,
                                target_entity_name=entity_name,
                                source_layer=EVENT_PHASE_BRONZE_ARMS,
                                target_layer=EVENT_PHASE_CLEANSE_ARMS,
                                retry_count=retry_count,
                                pipeline_id=pipeline_id,
                                started_at=act_start,
                                delta_version=None,
                                error_message=format_exception_message(e, message)
            )
           
            raise Exception(message) from e
         #end of try/except  
    #end of for loop

    #TODO FG *** Need to check what this quarantine process does
    if total_quarantined:
        print(f"  ({total_quarantined} total rows quarantined to {QUARANTINE_BASE_PATH})")
   
except Exception as e:
    e_msg = format_exception_message(e, str(e))
    captured_exception = Exception(e_msg)

# =============================================================================
# Log batch completed
# =============================================================================

final_status = EVENT_STATUS_SUCCEEDED
errormessage_str=None
message = f"ARMS cleanse has {final_status} for batch_run_id: {batch_run_id}"

if captured_exception is not None:
    final_status = EVENT_STATUS_FAILED
    message = f"ARMS cleanse has {final_status} for batch_run_id: {batch_run_id}"
    errormessage_str = format_exception_message(captured_exception, message)

log_batch_event(METADATA_AUDIT_PATH,
    batch_run_id=batch_run_id,
    run_date=run_date,
    retry_count=retry_count,
    event_type=EVENT_TYPE_COMPLETED,
    status=final_status,
    pipeline_name=PIPELINE_NAME,
    pipeline_id=pipeline_id,
    parent_pipeline_id=parent_pipeline_id,
    top_level_pipeline_id=top_level_pipeline_id,
    phase=EVENT_PHASE_CLEANSE_ARMS,
    error_message=errormessage_str
)

print(message)

if captured_exception is not None:
    raise Exception(message) from captured_exception 

StatementMeta(DataWHv35, 7628, 57, Finished, Available, Finished, False)

2026-11-10T04:06:49.847Z | INFO    | mdw_shared.audit | batch=- | pipeline=- | Batch event: started running pipeline=pl_coordinator_silver_arms_ingest batch=run_20261102T0007
Found 55 tables to ingest from arms
2026-11-10T04:06:53.335Z | INFO    | mdw_shared.audit | batch=- | pipeline=- | Activity started: asqa_registrationsubmission (id=f8dfeb0c-9708-4b58-bc68-378b2c8d9534) batch=run_20261102T0007
2026-11-10T04:06:55.655Z | INFO    | mdw_shared.metadata | batch=- | pipeline=- | Watermark updated: mapping_id=43 value=None
2026-11-10T04:06:56.656Z | INFO    | mdw_shared.audit | batch=- | pipeline=- | Activity completed: asqa_registrationsubmission (id=f8dfeb0c-9708-4b58-bc68-378b2c8d9534) batch=run_20261102T0007 status=succeeded
  ~ asqa_registrationsubmission: no incremental changes
2026-11-10T04:06:57.782Z | INFO    | mdw_shared.audit | batch=- | pipeline=- | Activity started: account (id=b6db90d5-16e6-4040-981d-a5bb0de5a7a7) batch=run_20261102T0007
2026-11-10T04:07:06.930Z | INFO    

In [24]:
df = spark.read.format("delta").load(source_path)
...
rows_read = df.count()
...
df.write.format("delta").mode("append").save(target_path)

StatementMeta(DataWHv35, 7628, 61, Finished, Available, Finished, False)

AnalysisException: [DELTA_PARTITION_COLUMN_NOT_FOUND] Partition column `mdw_batch_run_id` not found in schema [Id, SinkCreatedOn, SinkModifiedOn, name, ownerid, owneridtype, versionnumber, yominame, IsDelete, CreatedOn, createdonpartition, PartitionId]

In [22]:
display(
    spark.read.format("delta").load(
        "abfss://silver2@deintdlsaueaasqebla.dfs.core.windows.net/cleansed/arms/asqa_rmhistory"
    )
)

StatementMeta(DataWHv35, 7628, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c0679e8b-74b3-4dab-a2b5-224422e1a58c)

In [23]:
display(
    spark.read.format("delta").load(
        "abfss://silver2@deintdlsaueaasqebla.dfs.core.windows.net/cleansed/arms/asqa_studentmanagementprovider"
    )
)


StatementMeta(DataWHv35, 7628, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bd40cfc6-3231-4148-b51f-aad309e82960)